In [4]:
import json
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import time

from rich.progress import Progress

from python_magnetrun.MagnetRun import MagnetRun, load_mrun

instrumented = False
try:
    import pyinstrument
    p = pyinstrument.Profiler()
    p.start()
    instrumented = True
    print("!!! pyinstrument available, will profile !!!")
except   ImportError:
    print("!!! no pyinstrument available, will not profile !!!")
    pass

DATA_DIR = Path("../Data")
PUPITRE_ROOT = Path("~/LNCMIG-Data/records/srv-data-install").expanduser()
PUPITRE_DIR = Path("~/LNCMIG-Data/records/srv-data-install/M9").expanduser()
PIGBROTHER = Path("../pigbrother_2025/M10_Overview_251201-0909.tdms")
DB = "test-magnetdb.duckdb"

FIELD_THRESHOLD = 0.1

RuntimeError: There is already a profiler running. You cannot run multiple profilers in the same thread or async context, unless you disable async support.

In [ ]:
# Sample the field at 20 points -> representation of the field profile
def compute_field_signature(mdata, field, threshold):
    from python_magnetrun.signature import Signature
    
    signature = Signature.from_mdata(mdata, field, "t", threshold)
    
    return ",".join(signature.to_dict())

# HOUSING SUMMARY
### Load and merge the housing summary files


In [6]:
PATH_COLUMNS = [
    "overview", "archive", "pupitre", "default", "trigger", "spike",
    "hybrid_kHz", "hybrid_rms", "hybrid_trigger", "hybrid_vprocess",
    "pigbrother_runlog", "pupitre_runlog",
]

rows = []
for file in sorted(DATA_DIR.glob("*_summary-*.json")):

    print(f"Loading {file.name}")

    housing = file.stem.split("_")[0]
    year = int(file.stem[-4: ])

    with open(file, "r") as f:
        data = json.load(f)

    df = pd.json_normalize(data)

    for col in PATH_COLUMNS:
        df[col] = df[col].apply(lambda x: Path(x).name if x else x)

    df["housing"] = housing
    df["year"] = year

    rows.append(df)

summary_df = pd.concat(rows, ignore_index = True)

summary_df["experiment_id"]       = None
summary_df["field_max"]           = pd.Series(dtype = "float64")
summary_df["field_mean"]          = pd.Series(dtype = "float64")
summary_df["field_time_on"]       = pd.Series(dtype = "float64")
summary_df["mode"]                = ""
summary_df["field_signature"]     = ""
summary_df["reference_signature"] = ""

print(f"Found {len(summary_df)} summary files")
print(summary_df.head())

Loading M10_summary-2022.json
Loading M10_summary-2023.json
Loading M10_summary-2024.json
Loading M10_summary-2025.json
Loading M10_summary-2026.json
Loading M9_summary-2022.json
Loading M9_summary-2023.json
Loading M9_summary-2024.json
Loading M9_summary-2025.json
Loading M9_summary-2026.json
Found 1776 summary files
                   filename                       overview  \
0  M10_Overview_220204-1003  M10_Overview_220204-1003.tdms   
1  M10_Overview_220204-1510  M10_Overview_220204-1510.tdms   
2  M10_Overview_220206-1521  M10_Overview_220206-1521.tdms   
3  M10_Overview_220208-0951  M10_Overview_220208-0951.tdms   
4  M10_Overview_220211-0941  M10_Overview_220211-0941.tdms   

                        archive                    pupitre default trigger  \
0  M10_Archive_220204-1003.tdms  2022.02.04 - 10:04:03.txt                   
1  M10_Archive_220204-1510.tdms  2022.02.04 - 15:10:08.txt                   
2  M10_Archive_220206-1521.tdms  2022.02.06 - 15:21:53.txt               

### Refresh the housing summary table and display basic stats

In [7]:
con = duckdb.connect(DB)
con.execute(
    """
        DROP TABLE IF EXISTS housing_summary
    """
)
con.register("summary_df", summary_df)
con.execute(
    """
        CREATE TABLE housing_summary AS
        SELECT *
        FROM summary_df
    """
)

print("\nRows:\n",
    con.execute(
        """
            SELECT housing, year, COUNT(*) AS n
            FROM housing_summary
            GROUP BY housing, year
            ORDER BY housing, year
        """
    ).fetchdf()
)


Rows:
   housing  year    n
0     M10  2022  238
1     M10  2023  209
2     M10  2024  146
3     M10  2025  233
4     M10  2026   66
5      M9  2022  237
6      M9  2023  222
7      M9  2024  162
8      M9  2025  191
9      M9  2026   72


### Data quality audit

In [8]:
## Check the date schema
print("\nTABLE SCHEMA: ",
    con.execute(
        """
            DESCRIBE housing_summary
        """
    ).fetchdf()
)
## Count imported records
print("\nNUMBER OF ROWS:", 
    con.execute(
        """
            SELECT COUNT(*) FROM housing_summary
        """
    ).fetchone()[0]
)
# Count number of records linked to experiments
print("NUMBER OF MATCHES:",
    con.execute(
        """
            SELECT COUNT(*)
            FROM housing_summary AS h
            JOIN experiments AS e
            ON h.pupitre LIKE '%' || e.file
        """).fetchone()[0]
)


TABLE SCHEMA:              column_name column_type null   key default extra
0              filename     VARCHAR  YES  None    None  None
1              overview     VARCHAR  YES  None    None  None
2               archive     VARCHAR  YES  None    None  None
3               pupitre     VARCHAR  YES  None    None  None
4               default     VARCHAR  YES  None    None  None
5               trigger     VARCHAR  YES  None    None  None
6                 spike     VARCHAR  YES  None    None  None
7            hybrid_kHz     VARCHAR  YES  None    None  None
8            hybrid_rms     VARCHAR  YES  None    None  None
9        hybrid_trigger     VARCHAR  YES  None    None  None
10      hybrid_vprocess     VARCHAR  YES  None    None  None
11    pigbrother_runlog     VARCHAR  YES  None    None  None
12       pupitre_runlog     VARCHAR  YES  None    None  None
13              housing     VARCHAR  YES  None    None  None
14                 year      BIGINT  YES  None    None  None
15      

In [9]:
# Check for missing files
print("\nMISSING FILES:\n",
    con.execute(
        """
            SELECT
                SUM(CASE WHEN overview = '' THEN 1 ELSE 0 END) AS overview,
                SUM(CASE WHEN archive  = '' THEN 1 ELSE 0 END) AS archive,
                SUM(CASE WHEN pupitre  = '' THEN 1 ELSE 0 END) AS pupiter
            from housing_summary
        """
    ).fetchdf()
)
# Check for duplicate files
print("\nDUPLICATE FILENAMES:\n",
    con.execute(
        """
            SELECT filename, COUNT(*) AS n
            FROM housing_summary
            GROUP BY filename
            HAVING COUNT(*) > 1
            ORDER BY n DESC
        """
    ).fetchdf()
)


MISSING FILES:
    overview  archive  pupiter
0       0.0     23.0    149.0

DUPLICATE FILENAMES:
 Empty DataFrame
Columns: [filename, n]
Index: []


### Link with the user DB : Add foreign key column and populate

In [10]:
con.execute(
    """
        UPDATE housing_summary AS h
        SET experiment_id = e.id
        FROM experiments AS e
        WHERE h.pupitre LIKE '%' || e.file
    """
)

print("\nLINKED EXPERIMENTS:",
    con.execute(
        """
            SELECT COUNT(*)
            FROM housing_summary
            WHERE experiment_id IS NOT NULL
        """
    ).fetchone()[0]
)
print(
    con.execute(
        """
            SELECT experiment_id, filename, pupitre
            FROM housing_summary
            WHERE experiment_id IS NOT NULL
            LIMIT 10
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT h.experiment_id, e.name, e.file, h.pupitre
            FROM housing_summary AS h
            JOIN experiments AS e
            ON h.experiment_id = e.id
            LIMIT 10
        """
    ).fetchdf()
)

rows = con.execute(
    """
        SELECT rowid, housing, pupitre
        FROM housing_summary
        WHERE pupitre <> ''
    """
).fetchall()

print(f"rows to process: {len(rows)}")
print(f"Pupitre files to process: {len([r for r in rows if r[2] != ''])}")


LINKED EXPERIMENTS: 1593
   experiment_id                  filename                    pupitre
0           1203  M10_Overview_220204-1003  2022.02.04 - 10:04:03.txt
1           1204  M10_Overview_220204-1510  2022.02.04 - 15:10:08.txt
2           1205  M10_Overview_220206-1521  2022.02.06 - 15:21:53.txt
3           1206  M10_Overview_220208-0951  2022.02.08 - 09:51:33.txt
4           1207  M10_Overview_220211-0941  2022.02.11 - 09:41:25.txt
5           1208  M10_Overview_220211-1724  2022.02.11 - 17:24:52.txt
6           1208  M10_Overview_220211-1801  2022.02.11 - 17:24:52.txt
7           1208  M10_Overview_220211-1803  2022.02.11 - 17:24:52.txt
8           1208  M10_Overview_220211-1805  2022.02.11 - 17:24:52.txt
9           1208  M10_Overview_220211-1807  2022.02.11 - 17:24:52.txt
   experiment_id                   name                       file  \
0           1203  2022.02.04 - 10:04:03  2022.02.04 - 10:04:03.txt   
1           1204  2022.02.04 - 15:10:08  2022.02.04 - 15:10:08.t

In [11]:
print(
    con.execute("""
        DESCRIBE housing_summary
    """).fetchdf()
)

            column_name column_type null   key default extra
0              filename     VARCHAR  YES  None    None  None
1              overview     VARCHAR  YES  None    None  None
2               archive     VARCHAR  YES  None    None  None
3               pupitre     VARCHAR  YES  None    None  None
4               default     VARCHAR  YES  None    None  None
5               trigger     VARCHAR  YES  None    None  None
6                 spike     VARCHAR  YES  None    None  None
7            hybrid_kHz     VARCHAR  YES  None    None  None
8            hybrid_rms     VARCHAR  YES  None    None  None
9        hybrid_trigger     VARCHAR  YES  None    None  None
10      hybrid_vprocess     VARCHAR  YES  None    None  None
11    pigbrother_runlog     VARCHAR  YES  None    None  None
12       pupitre_runlog     VARCHAR  YES  None    None  None
13              housing     VARCHAR  YES  None    None  None
14                 year      BIGINT  YES  None    None  None
15        experiment_id 

In [12]:
start = time.perf_counter()
with Progress() as progress:
    task = progress.add_task("Updating field stats", total=len(rows))
    for rowid, housing, pupitre in rows:
        filename = Path(pupitre).name
        filepath = PUPITRE_ROOT / housing / filename
        progress.update(task, description=f"{housing}/{filename}")

        if not filepath.exists():
            progress.advance(task)
            continue

        try:
            md = load_mrun(str(filepath), housing=housing)
            mdata = md.getMData()
            df = mdata.Data

            # from python_magnetrun.analysis.config import AnalysisConfig
            # config = AnalysisConfig.for_housing(housing)
            # threshold = config.thresholds.get("Field")
            threshold = 1.e-3

            field = df["Field"]
            field_signature = compute_field_signature(mdata, "Field", threshold)

            con.execute(
                """
                    UPDATE housing_summary
                    SET 
                        field_max = ?,
                        field_mean = ?,
                        field_time_on = ?,
                        field_signature = ?
                    WHERE rowid = ?
                """, 
                (float(field.max()), float(field.mean()), int((field > FIELD_THRESHOLD).sum()), field_signature, int(rowid))
            )

        except Exception as e:
            progress.console.print(f"[red]{filename}: {e}[/red]")

        progress.advance(task)

end = time.perf_counter()
print(f"Dataframe updated in {int((end - start) // 60)} m {((end - start) % 60):.2f} s")

Output()

2022.02.04 - 10:04:03.txt: Signature.from_mdata() missing 1 required positional argument: 'threshold'

2022.02.04 - 15:10:08.txt: Signature.from_mdata() missing 1 required positional argument: 'threshold'

2022.02.06 - 15:21:53.txt: Signature.from_mdata() missing 1 required positional argument: 'threshold'

2022.02.08 - 09:51:33.txt: Signature.from_mdata() missing 1 required positional argument: 'threshold'

...

2026.05.07 - 15:58:14.txt: Signature.from_mdata() missing 1 required positional argument: 'threshold'

2026.05.08 - 09:00:22.txt: Signature.from_mdata() missing 1 required positional argument: 'threshold'

Duplicates found in _timestamp: 
/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M9/2026.05.09 - 08:23:57.txt — 367 
duplicate(s) removed

2026.05.09 - 08:23:57.txt: Signature.from_mdata() missing 1 required positional argument: 'threshold'

Dataframe updated in 25 m 18.42 s


In [10]:
# Validate update
print(
    con.execute(
        """
            SELECT COUNT(field_max) AS field_stats, COUNT(field_signature) AS signatures
            FROM housing_summary
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT experiment_id, field_max, field_signature
            FROM housing_summary
            WHERE field_signature <> ''
            LIMIT 10
        """
    ).fetchdf()
)

print(
    con.execute(
        """
            SELECT experiment_id, field_signature
            FROM housing_summary
            WHERE field_signature IS NOT NULL
            LIMIT 5
        """
    ).fetchdf()
)

   field_stats  signatures
0          849        1524
   experiment_id  field_max                                    field_signature
0           <NA>    29.9991  0.00,2.00,2.00,2.00,2.00,2.00,2.00,4.00,4.00,8...
1           <NA>    29.9991  0.00,2.00,2.00,2.00,2.00,2.00,2.00,4.00,4.00,8...
2           <NA>    29.9991  0.00,2.00,2.00,2.00,2.00,2.00,2.00,4.00,4.00,8...
3           <NA>     7.4665  0.00,0.00,0.54,0.68,1.18,2.88,4.59,6.29,6.81,7...
4           <NA>    30.1392  0.00,20.00,26.74,20.00,26.40,29.62,25.82,29.82...
5           <NA>    30.1392  0.00,20.00,26.74,20.00,26.40,29.62,25.82,29.82...
6           <NA>    30.1392  0.00,20.00,26.74,20.00,26.40,29.62,25.82,29.82...
7           <NA>    30.1392  0.00,20.00,26.74,20.00,26.40,29.62,25.82,29.82...
8           <NA>    30.1392  0.00,20.00,26.74,20.00,26.40,29.62,25.82,29.82...
9           <NA>    29.9991  0.00,0.01,0.01,0.01,9.82,21.67,24.26,26.48,28....
   experiment_id field_signature
0           <NA>                
1          

In [ ]:
con.close()

files = sorted(PUPITRE_DIR.glob("*.txt"))

print(f"Found {len(files)} files")
print(f"Loading: {files[0].name}")

md = load_mrun(str(files[0]))
df = md.getMData().Data

print("\nAVAILABLE CHANNELS:", md.getKeys())
print("\nCOLUMNS:", df.columns.tolist())
print("\nFIRST ROWS:\n", df.head())

Found 821 files
Loading: 2023.01.30 - 16:02:56.txt

AVAILABLE CHANNELS: ['Date', 'Time', 'Field', 'Tin1', 'Tin2', 'Tout', 'TAlimout', 'HP1', 'HP2', 'BP', 'Flow1', 'Flow2', 'Rpm1', 'Rpm2', 'Idcct1', 'Idcct2', 'Idcct3', 'Idcct4', 'Icoil1', 'Ucoil1', 'DRcoil1', 'Tcal1', 'Icoil2', 'Ucoil2', 'DRcoil2', 'Tcal2', 'Icoil3', 'Ucoil3', 'DRcoil3', 'Tcal3', 'Icoil4', 'Ucoil4', 'DRcoil4', 'Tcal4', 'Icoil5', 'Ucoil5', 'DRcoil5', 'Tcal5', 'Icoil6', 'Ucoil6', 'DRcoil6', 'Tcal6', 'Icoil7', 'Ucoil7', 'DRcoil7', 'Tcal7', 'Icoil8', 'Ucoil8', 'DRcoil8', 'Tcal8', 'Icoil9', 'Ucoil9', 'DRcoil9', 'Tcal9', 'Icoil10', 'Ucoil10', 'DRcoil10', 'Tcal10', 'Icoil11', 'Ucoil11', 'DRcoil11', 'Tcal11', 'Icoil12', 'Ucoil12', 'DRcoil12', 'Tcal12', 'Icoil13', 'Ucoil13', 'DRcoil13', 'Tcal13', 'Icoil14', 'Ucoil14', 'DRcoil14', 'Tcal14', 'Icoil15', 'Ucoil15', 'DRcoil15', 'Tcal15', 'Icoil16', 'Ucoil16', 'DRcoil16', 'Tcal16', 'Pmagnet', 'Ptot', 'teb', 'tsb', 'debitbrut', 'Q']

COLUMNS: ['Date', 'Time', 'Field', 'Tin1', 'Tin2', '

# MODE INFERRING

In [ ]:
mrun = load_mrun(str(PIGBROTHER), housing = "M10")
mdata = mrun.getMData()
print(mdata)

print("Courants_Alimentations columns:", mdata.Data["Courants_Alimentations"].columns)

magnetdata.fromtdms: ../pigbrother_2025/M10_Overview_251201-0909.tdms
magnetrun.fromtdms: start_time=2025-12-01 08:09:13.922483, type=<class 'datetime.datetime'>
MagnetData(Type=1, Groups={'Courants_Alimentations': {'Courant_A1': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A1', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A2': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A2', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A3': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A3', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A4': OrderedDict({'wf_start_time': np.datet

# PROPOSALS

In [ ]:
# Load proposals metadata and parse experiment date ranges

proposals_df = pd.read_csv(DATA_DIR / "proposals_2026-07-22_with_sites.csv")
proposals_df["Debut"] = pd.to_datetime(proposals_df["Debut"], errors = "coerce")
proposals_df["Fin"]   = pd.to_datetime(proposals_df["Fin"],   errors = "coerce")

In [ ]:
# Connect to the database and recreate the proposals table

con = duckdb.connect(DB)

con.execute(
    """
        DROP TABLE IF EXISTS proposals
    """
)
con.register("proposals_df", proposals_df)
con.execute(
    """
        CREATE TABLE proposals AS
        SELECT * FROM proposals_df
    """
)

In [ ]:
# Inspect imported proposal schema as well as the experiments table
 
print(
    con.execute(
        """
            DESCRIBE proposals
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT * FROM proposals 
            LIMIT 10
        """
    ).fetchdf()
)

print(
    con.execute(
        """
            DESCRIBE experiments
        """
    )
)
print(
    con.execute(
        """
            SELECT * FROM experiments
            LIMIT 10
        """
    ).fetchdf()
)


        column_name   column_type null   key default extra
0           Acronym       VARCHAR  YES  None    None  None
1         ProjectID        BIGINT  YES  None    None  None
2      ResearchArea       VARCHAR  YES  None    None  None
3          Facility       VARCHAR  YES  None    None  None
4      ProposalType       VARCHAR  YES  None    None  None
5        accessMode        DOUBLE  YES  None    None  None
6        CallNumber        BIGINT  YES  None    None  None
7                id        BIGINT  YES  None    None  None
8   ExperimentState       VARCHAR  YES  None    None  None
9              Site       VARCHAR  YES  None    None  None
10    ShotsHourDone        DOUBLE  YES  None    None  None
11       EnergyUsed        DOUBLE  YES  None    None  None
12            Debut  TIMESTAMP_NS  YES  None    None  None
13              Fin  TIMESTAMP_NS  YES  None    None  None
       Acronym  ProjectID ResearchArea  Facility ProposalType  accessMode  \
0    GMS06-217       3218           MS

In [ ]:
# Check temporal coverage of the proposal metadata

print(
    con.execute(
        """
            SELECT MIN(file), MAX(file), COUNT(*)
            FROM experiments
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT DISTINCT year
            FROM housing_summary
            ORDER BY year
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT MIN(Debut), MAX(Fin), COUNT(*)
            FROM proposals
        """
    ).fetchdf()
)

                   min(file)                  max(file)  count_star()
0  2025.03.13 - 15:14:35.txt  2026.04.27 - 13:27:14.txt           755
   year
0  2022
1  2023
2  2024
3  2025
4  2026
  min(Debut)   max(Fin)  count_star()
0 2009-01-19 2023-10-27          1712


In [ ]:
# Add proposal column to housing_summary unless it already exists

con.execute(
    """
        ALTER TABLE housing_summary
        ADD COLUMN IF NOT EXISTS proposal VARCHAR;
    """
)

# Link housing records to proposals by magnet site and experiment date

con.execute(
    """
        UPDATE housing_summary AS h
        SET proposal = p.Acronym
        FROM proposals AS p
        WHERE h.pupitre <> '' AND h.pupitre IS NOT NULL
            AND h.site = regexp_replace(p.Site, '[ie]$', '')
            AND strptime(right(replace(h.pupitre, '.txt', ''), 19), '%y.%m.%d - %H:%M:%S')
        BETWEEN CAST(p.Debut AS TIMESTAMP) AND CAST(p.Fin AS TIMESTAMP);
    """
)

In [ ]:
# Validate propsal linkage

print(
    con.execute(
        """
            SELECT COUNT(*) AS total, COUNT(proposal) AS linked
            FROM housing_summary;
        """
    ).fetchdf()
)

   total  linked
0   1524     463


In [47]:
con.close()

In [ ]:
proposals_df = pd.read_csv(DATA_DIR / "proposals_2026-07-22.csv")
proposals_df["Experiment Start Date"] = pd.to_datetime(proposals_df["Experiment Start Date"], errors = "coerce")
proposals_df["Experiment End Date"]   = pd.to_datetime(proposals_df["Experiment End Date"], errors = "coerce")

print(proposals_df[["Acronym", "Magnet Sites", "Experiment Start Date", "Experiment End Date"]].head(), proposals_df.shape)

     Acronym  Magnet Sites Experiment Start Date Experiment End Date
0  GIS01-226           NaN            2026-10-20          2026-10-25
1        NaN           NaN                   NaT                 NaT
2        NaN           NaN                   NaT                 NaT
3  GIS02-126           NaN                   NaT                 NaT
4        NaN           NaN                   NaT                 NaT (1912, 13)


In [ ]:
# Check Magent Sites in new proposals_2026-07-26.csv

print(proposals_df["Magnet Sites"].dtype)
print(len(proposals_df))
print(proposals_df["Magnet Sites"].notna().sum())

float64
1912
0


In [ ]:
###
print(
    con.execute("""
        SELECT year, COUNT(*)
        FROM housing_summary
        GROUP BY year
        ORDER BY year
    """).fetchdf()
)

if instrumented:
    p.stop()
    print(p.output_text(unicode=True, color=True))

NameError: name 'con' is not defined